# Vectorbt rewrite

This notebook replaces the backtesting.py workflow with a vectorbt-based EMA/VWAP backtest.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt

CASH = 80_000
COMMISSION = 0.00015
RISK_PER_TRADE = 0.0025

def load_data(file_path):
    # Load OHLCV CSV and attach a daily-reset VWAP column (running cumsum of typical_price*volume per calendar day)
    data = pd.read_csv(file_path, parse_dates=["Datetime"], index_col="Datetime")
    # Typical price: per-bar mean of High, Low, Close
    typical_price = (data["High"] + data["Low"] + data["Close"]) / 3
    # Volume-weighted contribution of each bar to the VWAP numerator
    tp_vol = typical_price * data["Volume"]
    # Calendar-day key used to reset the cumsum at each session boundary
    dates = data.index.date
    # Daily-reset VWAP: cumsum(tp_vol) / cumsum(Volume), grouped by date so it restarts at zero each day
    data["VWAP"] = (tp_vol.groupby(dates).cumsum() / data["Volume"].groupby(dates).cumsum()).astype(float)
    return data

def make_signals(data, n1=9, n2=21, atr_period=14, sl_mult=1.0, rr_ratio=0.5):
    close = data["Close"].astype(float)
    high = data["High"].astype(float)
    low = data["Low"].astype(float)

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    ema2 = vbt.indicators.MA.run(close, window=n2, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close
    size_value = (CASH * RISK_PER_TRADE) / sl_pct

    long_signal = (ema1.ma > ema2.ma) & (close > data["VWAP"]) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_signal = (ema2.ma > ema1.ma) & (close < data["VWAP"]) & np.isfinite(atr.atr) & (atr.atr > 0)

    return pd.DataFrame({
        "ema1": ema1.ma,
        "ema2": ema2.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "size_value": size_value.where(np.isfinite(size_value) & (size_value > 0), 0.0),
        "long": long_signal,
        "short": short_signal,
    })

def run_backtest(data, n1=9, n2=21, atr_period=14, sl_mult=1.0, rr_ratio=0.5):
    signals = make_signals(data, n1=n1, n2=n2, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy()
    short_signal = signals["short"].to_numpy()
    
    portfolio = vbt.Portfolio.from_signals(
        close=data["Close"].to_numpy(),
        entries=long_signal,
        exits=short_signal,
        short_entries=short_signal,
        short_exits=long_signal,
        init_cash=CASH,
        fees=COMMISSION,
        size=signals["size_value"].to_numpy(),
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(),
        tp_stop=signals["tp_pct"].to_numpy(),
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, n2, atr_period, sl_mult, rr_ratio):
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, n2=n2, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio)
        return {
            "n1": n1, "n2": n2, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "n2": n2, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [ ]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M5.csv").expanduser()
data = load_data(file_path)

portfolio, stats, signals = run_backtest(data, rr_ratio=1)
print(stats)
# portfolio.plot()
trades = portfolio.trades.records_readable
print(trades)

In [ ]:
db_path = Path("backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS v1_rr_vectorbt (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    cagr_pct REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

In [ ]:
asset = "XAUUSD"
timeframe = "M5"
file_path = Path(f"~/market_data/{asset}/{asset}_{timeframe}.csv").expanduser()
data = load_data(file_path)
portfolio, stats, signals = run_backtest(data)

row = (
    asset,
    timeframe,
    9,
    21,
    float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
    float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
    float(stats["CAGR [%]"]) if stats.get("CAGR [%]") is not None else None,
    float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
    float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
    float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
    int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
    float(stats["End Value"]) if stats.get("End Value") is not None else None,
)

cursor.execute(
    "INSERT INTO v1_rr_vectorbt (asset, timeframe, n1, n2, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    row,
)
conn.commit()
stats

## Optimization

In [ ]:

from itertools import product
import time


TABLE_NAME = "test"
db_path = Path("backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15), # fast ema
    range(18, 30), # slow ema
    range(10, 21), # atr period
    [0.5, 1.0, 1.5, 2.0], # sl mult
    [0.25, 0.5, 0.75, 1.0, 1.5, 2.0], # rr ratio
))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
for i, (timeframe, n1, n2, ap, sm, rr) in enumerate(grid):

    asset = "XAUUSD"
    file_path = Path(f"~/market_data/{asset}/{asset}_{timeframe}.csv").expanduser()
    data = load_data(file_path)

    t1 = time.time()
    result = run_params(data, n1, n2, ap, sm, rr)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, n2, atr_period, sl_mult, rr_ratio, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["n2"], result["atr_period"], result["sl_mult"], result["rr_ratio"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    print(f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} n2={n2} ap={ap} sm={sm} rr={rr} | sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s)")
    if (i + 1) % 100 == 0:
        print(f"[{time.strftime('%H:%M:%S')}] {i+1}/{len(grid)} done in {time.time()-t0:.1f}s")

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)

In [ ]:

from itertools import product
import time


TABLE_NAME = "test_paired"
db_path = Path("backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    # ["M1", "M5", "M15"], # timeframes
    ["M1"], # timeframes
    range(5, 15), # fast ema
    range(18, 30), # slow ema
    [14], # atr period
    [1], # sl mult
    [1], # rr ratio
))

t0 = time.time()
print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
for i, (timeframe, n1, n2, ap, sm, rr) in enumerate(grid):

    asset = "XAUUSD"
    file_path = Path(f"~/market_data/{asset}/{asset}_{timeframe}.csv").expanduser()
    data = load_data(file_path)

    result = run_params(data, n1, n2, ap, sm, rr)
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, n2, atr_period, sl_mult, rr_ratio, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["n2"], result["atr_period"], result["sl_mult"], result["rr_ratio"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    if (i + 1) % 100 == 0:
        print(f" {i+1}/{len(grid)} done in {time.time()-t0:.1f}s")

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df[sweep_df["num_trades"] >= 1000]
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null & trades>=1000: {len(sweep_df)}")
sweep_df.head(20)

pair_groups = sweep_df.groupby(["n1", "n2"]).agg(
    sharpe_mean=("sharpe_ratio", "mean"),
    expectancy_mean=("expectancy", "mean"),
    return_pct_mean=("return_pct", "mean"),
    max_dd_mean=("max_dd", "mean"),
    win_rate_mean=("win_rate", "mean"),
    profit_factor_mean=("profit_factor", "mean"),
    trades_mean=("num_trades", "mean"),
    count=("sharpe_ratio", "size"),
).reset_index()

best_sharpe = pair_groups.sort_values("sharpe_mean", ascending=False).head(10)
print("\nTop pairs by Sharpe Ratio (mean across timeframes):")
print(best_sharpe)

best_expectancy = pair_groups.sort_values("expectancy_mean", ascending=False).head(10)
print("\nTop pairs by Expectancy (mean across timeframes):")
print(best_expectancy)

pair_groups["sharpe_rank"] = pair_groups["sharpe_mean"].rank(ascending=False)
pair_groups["expectancy_rank"] = pair_groups["expectancy_mean"].rank(ascending=False)
pair_groups["combined_rank"] = pair_groups["sharpe_rank"] + pair_groups["expectancy_rank"]
best_combined = pair_groups.sort_values("combined_rank").head(10)
print("\nTop pairs by combined Sharpe + Expectancy rank:")
print(best_combined)


[09:29:06] Testing 120 combinations -> test_paired
